# GGUF offload metadata stream patcher

This notebook takes the configured GGUF shards from
the configured source Hugging Face repo/folder, injects StorageLLM JUJU v2
QKV/offload metadata into each GGUF header, uploads each patched shard to one
target Hugging Face model repo, then deletes local output files before
continuing.

The loop is intentionally one-file-at-a-time:

1. read the source GGUF through HTTP Range requests,
2. stream-copy it into one patched local GGUF with `offload.*` metadata KVs,
3. upload the patched GGUF, verify JSON, and portable metadata exports,
4. delete local source/output/export files,
5. run GC and continue to the next shard.

Persistent FP16/BF16 plain KV cache is disabled in the metadata contract. QKV
packed cache is the required runtime path. The original model note about
model-specific runtime flags can be provided through `REQUIRED_RUNTIME_FLAGS`.

For a new GGUF variant, edit only the `CHANGE HERE` block at the top of the
first code cell. In the common case set `SOURCE_HF_REPO_ID`, `SOURCE_SUBDIR`,
and `TARGET_HF_OWNER` or `TARGET_HF_REPO_ID`; leave `SOURCE_FILE_PREFIX` and
`SOURCE_FILE_COUNT` blank so the notebook discovers shards automatically.


In [ ]:
import gc
import hashlib
import io
import json
import math
import os
import re
import shutil
import struct
import subprocess
import sys
import time
import urllib.parse
import urllib.request
from getpass import getpass
from pathlib import Path

NOTEBOOK_BUILD_ID = "gguf-offload-metadata-stream-patch-20260430"

# ========================= CHANGE HERE =========================
# Only these values should normally be changed for another GGUF folder.
# SOURCE_FILE_PREFIX and SOURCE_FILE_COUNT can stay blank; the notebook will
# discover shard names from Hugging Face and derive both values automatically.
SOURCE_HF_REPO_ID = os.environ.get("SOURCE_HF_REPO_ID", "unsloth/GLM-5.1-GGUF")
SOURCE_SUBDIR = os.environ.get("SOURCE_SUBDIR", "UD-IQ2_M").strip("/")
SOURCE_FILE_PREFIX = os.environ.get("SOURCE_FILE_PREFIX", "").strip()
SOURCE_FILE_COUNT_TEXT = os.environ.get("SOURCE_FILE_COUNT", "").strip()
SOURCE_FILE_EXT = os.environ.get("SOURCE_FILE_EXT", ".gguf")
if not SOURCE_FILE_EXT.startswith("."):
    SOURCE_FILE_EXT = "." + SOURCE_FILE_EXT
TARGET_SUBDIR = os.environ.get("TARGET_SUBDIR", SOURCE_SUBDIR).strip("/")
TARGET_HF_OWNER = os.environ.get("TARGET_HF_OWNER", "storagejuju").strip()
TARGET_HF_REPO_ID = os.environ.get("TARGET_HF_REPO_ID", "").strip()
# ===============================================================

BASE_DOWNLOAD_URL = f"https://huggingface.co/{SOURCE_HF_REPO_ID}/resolve/main"

MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", "/content/GGUF_OFFLOAD_PATCH"))
WORK_DIR = Path(os.environ.get("JUJU_V2_WORK_DIR", "/content/gguf_offload_patch_work"))
IN_DIR = WORK_DIR / "input"
OUT_DIR = WORK_DIR / "output"
VERIFY_DIR = MODEL_ROOT / "verify"
EXPORT_DIR = MODEL_ROOT / "offload_exports"
GGUF_EXPORT_DIR = EXPORT_DIR / "gguf"
SAFETENSORS_EXPORT_DIR = EXPORT_DIR / "safetensors"
SIDECAR_EXPORT_DIR = EXPORT_DIR / "sidecar"
CLEAN_LOCAL_WORK_ON_START = os.environ.get("CLEAN_LOCAL_WORK_ON_START", "1") != "0"
if CLEAN_LOCAL_WORK_ON_START:
    for path in (IN_DIR, OUT_DIR, VERIFY_DIR, EXPORT_DIR):
        if path.exists():
            shutil.rmtree(path)
for path in (IN_DIR, OUT_DIR, VERIFY_DIR, GGUF_EXPORT_DIR, SAFETENSORS_EXPORT_DIR, SIDECAR_EXPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)

def repo_path_join(folder, name):
    return f"{folder}/{name}" if folder else name


def hf_tree_api_url(repo_id, folder):
    repo = urllib.parse.quote(repo_id, safe="/")
    if folder:
        return f"https://huggingface.co/api/models/{repo}/tree/main/{urllib.parse.quote(folder, safe='/')}"
    return f"https://huggingface.co/api/models/{repo}/tree/main"


def env_or_colab_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    return value or ""


def list_hf_gguf_names(repo_id, folder, ext):
    request = urllib.request.Request(hf_tree_api_url(repo_id, folder))
    token = env_or_colab_secret("SOURCE_HF_TOKEN") or env_or_colab_secret("HF_TOKEN")
    if token:
        request.add_header("Authorization", f"Bearer {token}")
    with urllib.request.urlopen(request, timeout=120) as response:
        entries = json.loads(response.read().decode("utf-8"))
    names = []
    prefix = f"{folder}/" if folder else ""
    for entry in entries:
        path = str(entry.get("path") or "")
        kind = str(entry.get("type") or "")
        if kind and kind != "file":
            continue
        if not path.lower().endswith(ext.lower()):
            continue
        name = path[len(prefix):] if prefix and path.startswith(prefix) else Path(path).name
        if "/" not in name:
            names.append(name)
    names.sort()
    return names


def derive_shard_prefix_and_count(names):
    pattern = re.compile(r"^(?P<prefix>.+)-(?P<idx>\d{5})-of-(?P<count>\d{5})" + re.escape(SOURCE_FILE_EXT) + r"$")
    parsed = []
    for name in names:
        match = pattern.match(name)
        if not match:
            return "", 0
        parsed.append((match.group("prefix"), int(match.group("idx")), int(match.group("count"))))
    if not parsed:
        return "", 0
    prefixes = {x[0] for x in parsed}
    counts = {x[2] for x in parsed}
    if len(prefixes) != 1 or len(counts) != 1:
        return "", 0
    count = next(iter(counts))
    indexes = sorted(x[1] for x in parsed)
    if indexes != list(range(1, count + 1)):
        return "", 0
    return next(iter(prefixes)), count


def build_source_file_specs():
    global SOURCE_FILE_PREFIX
    if SOURCE_FILE_PREFIX and SOURCE_FILE_COUNT_TEXT:
        count = int(SOURCE_FILE_COUNT_TEXT)
        names = [
            f"{SOURCE_FILE_PREFIX}-{idx:05d}-of-{count:05d}{SOURCE_FILE_EXT}"
            for idx in range(1, count + 1)
        ]
    else:
        names = list_hf_gguf_names(SOURCE_HF_REPO_ID, SOURCE_SUBDIR, SOURCE_FILE_EXT)
        if not names:
            raise RuntimeError(f"no {SOURCE_FILE_EXT} files found in {SOURCE_HF_REPO_ID}/{SOURCE_SUBDIR}")
        detected_prefix, detected_count = derive_shard_prefix_and_count(names)
        if detected_prefix and detected_count:
            SOURCE_FILE_PREFIX = SOURCE_FILE_PREFIX or detected_prefix
            names = [
                f"{SOURCE_FILE_PREFIX}-{idx:05d}-of-{detected_count:05d}{SOURCE_FILE_EXT}"
                for idx in range(1, detected_count + 1)
            ]
        else:
            SOURCE_FILE_PREFIX = SOURCE_FILE_PREFIX or Path(names[0]).stem
    specs = []
    count = len(names)
    for idx, name in enumerate(names, start=1):
        specs.append({
            "index": idx,
            "count": count,
            "name": name,
            "source_path": repo_path_join(SOURCE_SUBDIR, name),
        })
    return specs


GGUF_FILE_SPECS = build_source_file_specs()
SOURCE_FILE_COUNT = len(GGUF_FILE_SPECS)
if not TARGET_HF_REPO_ID:
    safe_prefix = re.sub(r"[^A-Za-z0-9._-]+", "-", SOURCE_FILE_PREFIX).replace("_", "-").strip("-")
    TARGET_HF_REPO_ID = f"{TARGET_HF_OWNER}/{safe_prefix}-Offload"

START_FILE_INDEX = int(os.environ.get("START_FILE_INDEX", os.environ.get("START_PART", "1")))
END_FILE_INDEX = int(os.environ.get("END_FILE_INDEX", os.environ.get("END_PART", str(SOURCE_FILE_COUNT))))
FILES_TO_RUN = [
    spec for spec in GGUF_FILE_SPECS
    if START_FILE_INDEX <= int(spec["index"]) <= END_FILE_INDEX
]

UPLOAD_TO_HF = True
EXPORT_FOREIGN_METADATA = True
UPLOAD_FOREIGN_METADATA = True
DELETE_LOCAL_AFTER_UPLOAD = True
KEEP_FAILED_LOCAL_FILES = True
CONTINUE_ON_PART_ERROR = os.environ.get("CONTINUE_ON_PART_ERROR", "1") != "0"
PROGRESS_INTERVAL_BYTES = int(os.environ.get("PROGRESS_INTERVAL_BYTES", str(256 * 1024 * 1024)))
HTTP_CHUNK_BYTES = int(os.environ.get("HTTP_CHUNK_BYTES", str(16 * 1024 * 1024)))
GGUF_HEADER_RANGE_BYTES = int(os.environ.get("GGUF_HEADER_RANGE_BYTES", str(256 * 1024 * 1024)))
SMALL_FILE_DIRECT_GET_LIMIT_BYTES = int(os.environ.get("SMALL_FILE_DIRECT_GET_LIMIT_BYTES", str(512 * 1024 * 1024)))
GGUF_ARCH_META_PREFIXES = tuple(
    x.strip()
    for x in os.environ.get(
        "GGUF_ARCH_META_PREFIXES",
        "general.,split.,llama.,glm.,qwen.,qwen2.,qwen3.,qwen2moe.,deepseek2.,gemma.,mpt.,gptneox.",
    ).split(",")
    if x.strip()
)
BASE_MODEL_REPO = os.environ.get("BASE_MODEL_REPO", "")
DEFAULT_REQUIRED_RUNTIME_FLAGS = "--disable-shared-experts-fusion" if "GLM" in SOURCE_FILE_PREFIX.upper() else ""
REQUIRED_RUNTIME_FLAGS = [
    x.strip()
    for x in os.environ.get("REQUIRED_RUNTIME_FLAGS", DEFAULT_REQUIRED_RUNTIME_FLAGS).split(",")
    if x.strip()
]
SHARED_EXPERTS_FUSION_ALLOWED = os.environ.get("SHARED_EXPERTS_FUSION_ALLOWED", "0" if REQUIRED_RUNTIME_FLAGS else "1") == "1"
QKV_K_BITS = int(os.environ.get("QKV_K_BITS", "3"))
QKV_V_BITS = int(os.environ.get("QKV_V_BITS", "2"))
QKV_GROUP_SIZE = int(os.environ.get("QKV_GROUP_SIZE", "64"))
QKV_PAGE_SIZE_TOKENS = int(os.environ.get("QKV_PAGE_SIZE_TOKENS", "16"))
QKV_OUTLIER_CHANNELS = int(os.environ.get("QKV_OUTLIER_CHANNELS", "32"))
QKV_OUTLIER_BITS = int(os.environ.get("QKV_OUTLIER_BITS", "3"))

def infer_source_weight_quantization(*values):
    text = " ".join(str(v or "") for v in values).lower().replace("-", "_")
    checks = [
        ("ud_iq2", "gguf_iq2_m", 2, "gguf_iq2"),
        ("iq2", "gguf_iq2", 2, "gguf_iq2"),
        ("iq3", "gguf_iq3", 3, "gguf_iq3"),
        ("iq4", "gguf_iq4", 4, "gguf_iq4"),
        ("mxfp4", "gguf_mxfp4", 4, "gguf_mxfp4"),
        ("nvfp4", "gguf_nvfp4", 4, "gguf_nvfp4"),
        ("q8", "gguf_q8", 8, "gguf_q8"),
        ("q6", "gguf_q6", 6, "gguf_q6"),
        ("q5", "gguf_q5", 5, "gguf_q5"),
        ("q4", "gguf_q4", 4, "gguf_q4"),
        ("q3", "gguf_q3", 3, "gguf_q3"),
        ("q2", "gguf_q2", 2, "gguf_q2"),
    ]
    for needle, family, bits, kernel in checks:
        if needle in text:
            return {
                "family": family,
                "bits": bits,
                "kernel_family": kernel,
                "block_size": 256 if "k" in text or "iq" in text else 0,
                "dispatch_key": f"{kernel}:{bits}",
                "runtime_rule": "select_backend_kernel_by_quant_family_not_by_bits_only",
            }
    return {
        "family": "runtime_read_from_gguf_tensor_type",
        "bits": 0,
        "kernel_family": "runtime_read_from_gguf_tensor_type",
        "block_size": 0,
        "dispatch_key": "runtime_detect",
        "runtime_rule": "parse_each_gguf_tensor_type_before_kernel_selection",
    }


SOURCE_WEIGHT_QUANT = infer_source_weight_quantization(SOURCE_SUBDIR, SOURCE_FILE_PREFIX)

MODEL_PROFILE = {
    "model_id": os.environ.get("MODEL_ID", SOURCE_FILE_PREFIX.lower().replace("/", "_").replace("-", "_")),
    "artifact_stem": SOURCE_FILE_PREFIX,
    "source_repo": SOURCE_HF_REPO_ID,
    "source_subdir": SOURCE_SUBDIR,
    "source_file_prefix": SOURCE_FILE_PREFIX,
    "source_file_count": SOURCE_FILE_COUNT,
    "target_repo": TARGET_HF_REPO_ID,
    "target_subdir": TARGET_SUBDIR,
    "base_model_repo": BASE_MODEL_REPO,
    "arch_meta": {
        "model_family": "runtime_read_from_gguf_general.architecture",
        "n_layers": None,
        "moe_layers": "runtime_read_or_infer_from_gguf_tensor_names",
        "experts_per_moe_layer": "runtime_read_or_infer_from_gguf_tensor_names",
        "routed_experts_per_token": "runtime_read_or_model_config",
        "hidden_dim": None,
        "expert_intermediate_dim": None,
        "n_heads": None,
        "n_kv_heads": None,
        "head_dim": 128,
        "vocab_size": None,
        "max_seq_len": None,
        "attention_type": "runtime_read_from_gguf",
        "norm_type": "runtime_read_from_gguf_or_kernel_probe",
        "activation": "runtime_read_from_gguf_or_kernel_probe",
        "positional_encoding": "runtime_read_from_gguf",
        "tie_embeddings": "runtime_read_from_gguf",
    },
    "qkv_cache_schema": {
        "standard": "qkv_packed",
        "plain_kv_persistent_storage": False,
        "plain_kv_fallback": "debug_only_outside_hot_path",
        "append_path": "projection_scratch_to_qkv_page_append",
        "attention_path": "packed_qkv_attention",
        "offload_unit": "qkv_page",
        "k_bits": QKV_K_BITS,
        "v_bits": QKV_V_BITS,
        "group_size": QKV_GROUP_SIZE,
        "page_size_tokens": QKV_PAGE_SIZE_TOKENS,
        "append_granularity_tokens": 1,
        "layout": "layer_kvhead_page_token_dim",
        "rotation": {"enabled": True, "type": "hadamard_signs_or_materialized_matrix", "seed": 1234},
        "lloyd_max": {"enabled": True, "distribution": "beta_gaussian_approx", "store_codebook": True},
        "qjl": {"enabled": True, "seed": 5678, "materialize_matrix": False},
        "outlier": {"enabled": True, "channels": QKV_OUTLIER_CHANNELS, "bits": QKV_OUTLIER_BITS, "selection": "profile_or_static_first_n"},
        "residency": {
            "hot": "vram",
            "warm": "pinned_ram",
            "cold": "nvme_mmap",
            "eviction_policy": "sink_recent_h2o_score",
            "sink_tokens": 4,
            "recent_ratio": 0.10,
            "heavy_hitter_ratio": 0.10,
        },
    },
    "weight_quant_schema": {
        "packed_weight_policy": "preserve_source_gguf_quantization",
        "scale_policy": "runtime_read_from_gguf_tensor_types_and_blocks",
        "source_family": SOURCE_WEIGHT_QUANT["family"],
        "source_bits": SOURCE_WEIGHT_QUANT["bits"],
        "source_kernel_family": SOURCE_WEIGHT_QUANT["kernel_family"],
        "source_block_size": SOURCE_WEIGHT_QUANT["block_size"],
        "dispatch_key": SOURCE_WEIGHT_QUANT["dispatch_key"],
        "runtime_dispatch_rule": SOURCE_WEIGHT_QUANT["runtime_rule"],
        "backend_policy": {
            "required": "quant_family_specific_dequant_matmul_or_safe_rejection",
            "bits_are_not_sufficient": True,
            "iq_formats_require_importance_quant_metadata": True,
            "mxfp4_requires_mxfp4_table_not_nvfp4_table": True,
            "fallback": "do_not_route_unknown_quant_to_fp4_kernel",
        },
        "expert_bundle_unit": ["gate_proj", "up_proj", "down_proj"],
        "shared_experts_policy": {
            "dtype": "runtime_read_from_gguf",
            "fuse_with_routed_experts": SHARED_EXPERTS_FUSION_ALLOWED,
            "required_runtime_flags": REQUIRED_RUNTIME_FLAGS,
            "reason": "model_specific_policy_from_env_or_source_notes",
        },
    },
    "residency_policy": {
        "strategy": "active_set_vram",
        "anchors": {"embed": True, "lm_head": True, "final_norm": True, "layers": [0, 1]},
        "floating_layers": "runtime_derive_from_arch_meta_n_layers_excluding_anchors",
        "tier_order": ["vram", "pinned_ram", "nvme_mmap"],
        "valid_split_points": "runtime_derive_from_layer_count_and_memory_probe",
        "reentry_points": "runtime_derive_from_layer_count_and_memory_probe",
        "restore_priority_order": [
            "current_router_selected_expert_triplets",
            "lookahead_expert_triplets",
            "attention_current_layer",
            "anchors",
        ],
        "evict_priority_order": [
            "floating_ffn_down_proj",
            "late_attention_kv_related",
            "floating_ffn_gate_up",
            "early_attention",
            "anchors",
        ],
    },
    "prefetch_plan_hints": {
        "execution_order": "layer_major",
        "moe_unit": "expert_triplet_if_moe_else_dense_layer_block",
        "lookahead_window_layers_default": 2,
        "dedupe_key": ["layer", "expert"],
        "priority_classes": ["router_selected", "lookahead", "topology", "cold"],
    },
    "kernel_hints": {
        "required": ["packed_qkv_attention", "qkv_page_append", "residency_planner"],
        "preferred": ["quant_family_dequant_matmul", "scale_or_block_metadata_fusion", "async_prefetch"],
        "debug_only": ["plain_float_kv_attention"],
        "accumulate_dtype": "fp32_for_attention",
    },
    "execution_hints": {
        "backend_specific_terms_allowed": False,
        "async_transfer": {
            "enabled": True,
            "channel_count": 2,
            "channel_roles": ["compute", "host_device_transfer"],
            "channel_semantics": "independent_progress_preferred",
        },
        "pipeline_barriers": {
            "points": [4, 8, 16, 24],
            "type": "dependency_only",
            "full_device_sync_required": False,
        },
        "static_subgraph": {
            "capturable_ranges": [[0, 7], [16, 23], [32, 39], [48, 55]],
            "condition": {"min_repeat_count": 10, "static_shape_required": True},
            "runtime_translation": "backend_selects_graph_command_buffer_or_function_cache",
        },
        "fast_scratchpad": {
            "required_bytes_per_layer": 49152,
            "scope": "kernel_or_threadgroup_local",
        },
        "matrix_accelerator": {
            "preferred": True,
            "fallback": "simd",
        },
        "simd_width_hint": 32,
    },
    "dynamic_swap_triggers": [
        {
            "level": 1,
            "condition": {"accelerator_memory_pressure_gt": 0.80},
            "action": "old_qkv_pages_to_pinned_ram",
            "hysteresis": 0.05,
        },
        {
            "level": 2,
            "condition": {"accelerator_memory_pressure_gt": 0.88},
            "action": "floating_weight_blocks_to_host_memory",
            "compare_recompute_cost": True,
        },
        {
            "level": 3,
            "condition": {"accelerator_memory_pressure_gt": 0.94},
            "action": "qkv_pages_to_storage_and_floating_weights_to_host",
            "aggressive": True,
        },
        {
            "level": 4,
            "condition": {"accelerator_memory_pressure_gt": 0.98},
            "action": "force_qkv_page_eviction_and_emergency_residency_compaction",
            "qkv_force_evict_ratio": 0.30,
        },
    ],
    "memory_management_hints": {
        "pooling": {
            "accelerator_pool": {
                "preallocate": "runtime_decides",
                "slab_classes_bytes": [67108864, 268435456, 536870912, 1073741824, 2147483648],
                "eviction_policy": "lru_plus_hot_score",
                "defrag_threshold": 0.30,
            },
            "host_pool": {
                "prefer_page_locked": True,
                "prefer_large_pages": True,
                "numa_policy": "runtime_local",
            },
            "storage_pool": {
                "alignment_bytes": 4096,
                "preallocate": "runtime_decides",
            },
        },
        "buffer_reuse": {
            "alias_classes": [
                ["ffn_gate_output", "ffn_intermediate"],
                ["attention_output", "residual_input_when_safe"],
            ],
            "inplace_ops": ["rmsnorm", "rope", "silu"],
            "inter_layer_reuse": "output_input_pointer_handoff_when_shape_equal",
        },
        "transfer_compression": {
            "enabled": "runtime_decides_by_bandwidth_and_entropy",
            "host_to_accelerator": {"preferred_family": "fast_lz", "min_size_bytes": 67108864},
            "storage_to_host": {"preferred_family": "fast_zstd", "min_size_bytes": 67108864},
            "skip_when_already_entropy_dense": True,
        },
    },
    "cpu_execution_hints": {
        "capability_classes_preferred": ["matrix_tile_accelerator", "wide_simd", "numa_local_memory"],
        "threading": {
            "thread_count": "runtime_decides",
            "affinity": "performance_cores_preferred",
            "oversubscribe_io_threads": False,
        },
        "numa_hints": {
            "enabled": True,
            "node_selection": "runtime_local_to_active_accelerator_or_storage_path",
            "remote_access": "avoid_unless_required",
            "page_locked_allocation": "prefer_local_node",
            "worker_affinity": "bind_io_and_compute_workers_to_local_node_when_known",
        },
        "cache_blocking": {
            "l1": True,
            "l2": True,
            "software_prefetch_distance_bytes": 256,
        },
    },
    "kernel_config_hints": {
        "matmul": {
            "tile_m": 128,
            "tile_n": 64,
            "tile_k": 32,
            "target_occupancy": 0.75,
            "max_register_pressure": "medium",
        },
        "attention": {
            "tile_size": 128,
            "online_softmax": True,
            "accumulate_dtype": "fp32",
        },
        "packed_qkv": {
            "page_tokens": 16,
            "decode_directly_from_packed": True,
        },
    },
    "context_length_table": {
        "seq_1k": {"qkv_hot_ratio": 1.0, "weight_residency": "max_active_set", "batch_class": "large"},
        "seq_8k": {"qkv_hot_ratio": 0.70, "weight_residency": "active_set", "batch_class": "medium"},
        "seq_32k": {"qkv_hot_ratio": 0.30, "weight_residency": "reduced_active_set", "batch_class": "small"},
        "seq_128k": {"qkv_hot_ratio": 0.10, "weight_residency": "streaming", "batch_class": "single_or_micro"},
    },
    "io_schedule": {
        "backend_specific_terms_allowed": False,
        "queue_model": {
            "submission_queue_entries": 256,
            "completion_queue_entries": 512,
            "queue_depth_max": 128,
            "polling_thread_preferred": True,
            "polling_thread_affinity": "runtime_decides",
        },
        "backend_candidates": {
            "linux": "io_uring_if_available_else_pread_thread_pool",
            "windows": "overlapped_io_or_thread_pool",
            "macos": "dispatch_io_or_thread_pool",
            "portable_fallback": "bounded_thread_pool",
        },
        "priority_classes": {
            "anchor_tensors": "critical",
            "current_layer": "critical",
            "next_2_layers": "high",
            "lookahead_prefetch": "medium",
            "cold_restore": "low",
        },
        "deadline_policy": {
            "current_layer_deadline_ms": 15,
            "lookahead_deadline_ms": 50,
            "cold_deadline_ms": 250,
        },
        "runtime_translation": "native_async_io_submission_queue_or_threaded_fallback",
    },
    "streaming_attention": {
        "enabled": True,
        "mode": "qkv_paged_streaming",
        "window_size_tokens": 4096,
        "sink_tokens": 4,
        "chunked_attention": {
            "enabled": True,
            "chunk_size_tokens": 512,
            "overlap_tokens": 64,
        },
        "full_context_policy": "stream_qkv_pages_without_materializing_plain_kv",
    },
    "numerical_stability": {
        "global": {
            "attention_score_accumulate_dtype": "fp32",
            "softmax_method": "online",
            "qkv_dequant_accumulate_dtype": "fp32",
            "ffn_accumulate_dtype": "runtime_decides",
        },
        "per_role": {
            "attention": {"overflow_risk": "high_for_long_context", "softmax_method": "online"},
            "moe_expert": {"overflow_risk": "medium", "accumulate_dtype": "runtime_decides"},
            "lm_head": {"overflow_risk": "medium", "accumulate_dtype": "fp32_preferred"},
        },
        "per_layer_policy": {
            "layers_0_7": {"overflow_risk": "high", "attention_accumulate_dtype": "fp32"},
            "layers_8_63": {"overflow_risk": "medium", "attention_accumulate_dtype": "fp32"},
            "layers_64_78": {"overflow_risk": "high_for_long_context", "attention_accumulate_dtype": "fp32"},
        },
    },
    "continuous_batching": {
        "enabled": True,
        "max_concurrent_requests": "runtime_decides",
        "priority_classes": ["realtime", "interactive", "batch"],
        "prefix_caching": {
            "enabled": True,
            "cache_unit": "qkv_page",
            "share_common_prefix_pages": True,
        },
        "preemption": {
            "policy": "swap_qkv_pages_or_recompute",
            "preserve_sink_tokens": True,
        },
        "scheduler": "runtime_decides",
    },
    "batch_schedule": {
        "prefill": {
            "strategy": "accelerator_preferred_chunked",
            "chunked_prefill": True,
            "chunk_size_tokens": 512,
            "max_prefill_tokens_per_chunk": 4096,
            "overlap_with_decode": True,
        },
        "decode": {
            "strategy": "split_compute_and_transfer",
            "token_micro_batch_size": 4,
            "continuous_batching": True,
            "qkv_append_unit": "token_or_microbatch",
        },
        "scheduler": {
            "policy": "runtime_decides",
            "fairness": "priority_with_aging",
            "preemption": "swap_qkv_pages_or_recompute",
        },
    },
    "tensor_layout_schema": {
        "default_alignment_bytes": 256,
        "default_page_alignment_bytes": 4096,
        "payload": "per_tensor_layout_padding_stride_when_known",
        "missing_payload_policy": "runtime_derives_from_footer_blocks",
        "qkv_cache_layout": "layer_kvhead_page_token_dim",
        "fields": ["memory_order", "padding_bytes", "stride", "packed_axis", "tile_shape"],
    },
    "integrity_schema": {
        "file_hash": "sha256",
        "footer_hash": "sha256",
        "block_hash": "preserve_existing_sha1_when_present",
        "layer_hash": "derived_from_block_hashes_when_possible",
        "verify_on_load": True,
        "verify_on_transfer": "runtime_decides",
        "corruption_action": "reload_from_storage_source",
    },
    "activation_stats_schema": {
        "supported": True,
        "payload": "optional_per_layer_histogram_and_outlier_ratios",
        "missing_payload_policy": "runtime_uses_conservative_quant_defaults",
    },
    "sparsity_schema": {
        "supported": True,
        "payload": "optional_per_tensor_sparsity_and_format",
        "runtime_rule": "ignore_when_backend_has_no_profitable_sparse_kernel",
    },
    "runtime_monitoring_hints": {
        "sample_interval_ms": 10,
        "metrics": [
            "accelerator_memory_used",
            "interconnect_bandwidth_used",
            "host_memory_used",
            "storage_queue_depth",
            "accelerator_compute_utilization",
            "host_core_utilization",
            "qkv_page_hit_rate",
            "swap_frequency",
        ],
        "adaptive_split": {
            "enabled": True,
            "adjustment_interval_tokens": 100,
            "max_shift_per_interval_layers": 2,
        },
    },
    "thermal_power_hints": {
        "policy": "sustain_throughput_not_burst_clock",
        "runtime_probe_required": True,
        "throttle_action": "reduce_prefetch_depth_or_shift_work_to_host",
    },
    "speculative_execution_schema": {
        "enabled": "optional",
        "draft_location_preference": "host_or_small_accelerator",
        "verify_location_preference": "primary_accelerator",
        "max_draft_tokens": 5,
        "prefetch_verify_during_draft": True,
        "rollback_cost_class": "low",
    },
    "activation_checkpoint_schema": {
        "strategy": "selective",
        "checkpoint_layers": [0, 8, 16, 24, 32, 40, 48, 56, 64, 72],
        "recompute_policy": "use_when_cheaper_than_transfer_or_storage",
        "checkpoint_dtype": "bf16",
        "location_preference": "host_memory",
    },
    "mig_hints": {
        "supported_schema": True,
        "partitioning": "optional_runtime_profile",
        "preferred_partition": "runtime_decides",
        "isolation_policy": "do_not_assume_shared_memory_between_partitions",
    },
    "profiling_measured_data": {
        "present": False,
        "schema": "optional_per_layer_compute_transfer_io_bottleneck_measurements",
        "runtime_update_allowed": True,
        "missing_payload_policy": "use_cost_model_then_write_runtime_profile_sidecar",
    },
    "multi_accelerator_detail": {
        "enabled": "runtime_decides",
        "topology_source": "runtime_probe",
        "topology_schema": {
            "accelerators": "runtime_filled_list",
            "links": "runtime_filled_bandwidth_latency_matrix",
            "direct_peer_access": "runtime_filled_bool_matrix",
            "shared_host_numa_nodes": "runtime_filled_mapping",
        },
        "link_classes": ["direct_peer", "host_interconnect", "storage_direct", "network_or_none"],
        "split_strategies_supported": ["tensor_parallel", "pipeline_parallel", "expert_parallel"],
        "communication_unit": "qkv_page_or_expert_triplet",
        "p2p_required": False,
        "fallback": "single_accelerator_active_set",
    },
    "foreign_container_export": {
        "gguf": {
            "strategy": "offload.metadata_v2_json_plus_hot_scalar_keys",
            "scalar_keys": [
                "offload.format_version",
                "offload.backend_neutral",
                "offload.weight_quant_family",
                "offload.weight_bits",
                "offload.weight_kernel_family",
                "offload.qkv_k_bits",
                "offload.qkv_v_bits",
                "offload.qkv_page_size_tokens",
            ],
            "large_blocks": "json_string_values_under_offload_namespace",
        },
        "safetensors": {
            "strategy": "strings_in___metadata___under_offload_namespace",
            "large_blocks": "json_dumps_per_block",
            "sidecar_when_header_too_large": True,
        },
        "sidecar": {
            "strategy": "model.offload.json_or_metadata_juju",
            "works_with": ["gguf", "safetensors", "pytorch_bin", "custom_parts"],
        },
    },
    "backend_translation_contract": {
        "format_expresses": "intent_and_resource_needs_only",
        "runtime_translates": {
            "async_transfer.channel_count": "streams_queues_or_thread_pools",
            "pipeline_barriers": "events_barriers_or_dependency_edges",
            "static_subgraph": "graphs_command_buffers_indirect_buffers_or_function_cache",
            "matrix_accelerator": "tensor_matrix_tile_units_or_simd",
            "io_schedule.queue_model": "native_async_io_or_threaded_io",
            "multi_accelerator_detail": "device_specific_topology_runtime_plan",
        },
    },
    "compatibility": {
        "min_runtime_version": "0.6.0",
        "required_features": [
            "qkv_packed_cache_default",
            "plain_kv_persistent_storage_disabled",
            "packed_qkv_attention",
            "runtime_hardware_probe",
            "active_set_residency",
        ],
        "optional_features": [
            "direct_storage_io",
            "static_subgraph_capture",
            "speculative_decode",
            "multi_accelerator",
        ],
    },
}

print("NOTEBOOK_BUILD_ID:", NOTEBOOK_BUILD_ID)
print("SOURCE_HF_REPO_ID:", SOURCE_HF_REPO_ID)
print("SOURCE_SUBDIR:", SOURCE_SUBDIR)
print("SOURCE_FILE_PREFIX:", SOURCE_FILE_PREFIX)
print("SOURCE_FILE_COUNT:", SOURCE_FILE_COUNT)
print("TARGET_HF_REPO_ID:", TARGET_HF_REPO_ID)
print("TARGET_SUBDIR:", TARGET_SUBDIR)
print("FILES_TO_RUN:", [spec["index"] for spec in FILES_TO_RUN])
print("WORK_DIR:", WORK_DIR)


In [ ]:
def ensure_hf_api():
    try:
        from huggingface_hub import HfApi, create_repo
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"])
        from huggingface_hub import HfApi, create_repo
    return HfApi, create_repo


def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
    if token:
        return token
    return getpass("HF write token. Add it to Colab Secrets as HF_TOKEN to avoid pasting: ").strip()


def gguf_verify_json_name(spec):
    return f"{Path(spec['name']).stem}.offload.verify.json"


def target_gguf_repo_path(spec):
    return repo_path_join(TARGET_SUBDIR, spec["name"])


def source_gguf_hub_url(spec):
    from huggingface_hub import hf_hub_url
    return hf_hub_url(
        repo_id=SOURCE_HF_REPO_ID,
        filename=spec["source_path"],
        repo_type="model",
    )


def portable_stem_for_gguf(spec):
    return Path(spec["name"]).stem


def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def json_compact(obj):
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"), sort_keys=True)


def beta_pdf(x, dim):
    if x <= -1.0 or x >= 1.0:
        return 0.0
    variance = 1.0 / float(dim)
    sigma = math.sqrt(variance)
    return math.exp(-(x * x) / (2.0 * variance)) / (sigma * math.sqrt(2.0 * math.pi))


def lloyd_max_codebook(bits, dim, max_iters=100):
    levels = 1 << int(bits)
    sigma = 1.0 / math.sqrt(float(dim))
    support_min = -1.0
    support_max = 1.0
    if bits == 2:
        centroids = [-1.51 * sigma, -0.453 * sigma, 0.453 * sigma, 1.51 * sigma]
        thresholds = [support_min, (-1.51 - 0.453) * 0.5 * sigma, 0.0, (0.453 + 1.51) * 0.5 * sigma, support_max]
        return {"bits": bits, "dim": dim, "centroids": centroids, "thresholds": thresholds, "threshold_encoding": "finite_normalized_support"}
    centroids = [(-3.5 + 7.0 * i / max(1, levels - 1)) * sigma for i in range(levels)]
    thresholds = [0.0] * (levels + 1)
    for _ in range(max_iters):
        thresholds[0] = support_min
        for i in range(1, levels):
            thresholds[i] = (centroids[i - 1] + centroids[i]) * 0.5
        thresholds[levels] = support_max
        converged = True
        for i in range(levels):
            steps = 1000
            lo = thresholds[i]
            hi = thresholds[i + 1]
            step = (hi - lo) / float(steps)
            sum_x = 0.0
            sum_w = 0.0
            for j in range(steps + 1):
                x = lo + j * step
                w = beta_pdf(x, dim)
                sum_x += x * w
                sum_w += w
            if sum_w > 1e-10:
                nxt = sum_x / sum_w
                if abs(nxt - centroids[i]) > 1e-6:
                    converged = False
                centroids[i] = nxt
        if converged:
            break
    return {"bits": bits, "dim": dim, "centroids": centroids, "thresholds": thresholds, "threshold_encoding": "finite_normalized_support"}


def xorshift64(value):
    value &= (1 << 64) - 1
    value ^= (value << 13) & ((1 << 64) - 1)
    value ^= value >> 7
    value ^= (value << 17) & ((1 << 64) - 1)
    return value & ((1 << 64) - 1)


def deterministic_signs(count, seed):
    value = int(seed) or 1
    signs = []
    for _ in range(int(count)):
        value = xorshift64(value)
        signs.append(1 if (value & 1) else -1)
    return signs


def build_qkv_aux(profile, head_dim_override=None):
    qkv = profile["qkv_cache_schema"]
    head_dim = int(head_dim_override or profile["arch_meta"].get("head_dim") or 128)
    bits = sorted({int(qkv["k_bits"]), int(qkv["v_bits"]), int(qkv["outlier"]["bits"])})
    return {
        "head_dim": head_dim,
        "codebooks": {str(bit): lloyd_max_codebook(bit, head_dim) for bit in bits},
        "rotation_aux": {
            "type": "hadamard_signs",
            "seed": int(qkv["rotation"]["seed"]),
            "signs": deterministic_signs(head_dim, int(qkv["rotation"]["seed"])),
        },
        "qjl_aux": {
            "type": "seed_only",
            "seed": int(qkv["qjl"]["seed"]),
            "materialized": False,
        },
        "outlier_channel_map": list(range(int(qkv["outlier"]["channels"]))),
    }


_QKV_AUX_CACHE = {}


def get_qkv_aux(head_dim=None):
    global _QKV_AUX_CACHE
    key = int(head_dim or MODEL_PROFILE["arch_meta"].get("head_dim") or 128)
    if key not in _QKV_AUX_CACHE:
        _QKV_AUX_CACHE[key] = build_qkv_aux(MODEL_PROFILE, head_dim_override=key)
    return _QKV_AUX_CACHE[key]


def build_gguf_kv_entries(contract):
    qkv = contract["qkv_cache_schema"]
    residency = contract["residency_policy"]
    execution = contract["execution_hints"]
    scalar_entries = {
        "offload.format_version": ("uint32", 2),
        "offload.backend_neutral": ("bool", True),
        "offload.async_channel_count": ("uint32", int(execution["async_transfer"]["channel_count"])),
        "offload.simd_width_hint": ("uint32", int(execution.get("simd_width_hint", 0))),
        "offload.weight_quant_family": ("string", str(contract["source_weight_quant_family"])),
        "offload.weight_kernel_family": ("string", str(contract["source_weight_kernel_family"])),
        "offload.weight_bits": ("uint32", int(contract["source_weight_bits"])),
        "offload.weight_block_size": ("uint32", int(contract["source_weight_block_size"])),
        "offload.qkv_k_bits": ("uint32", int(qkv["k_bits"])),
        "offload.qkv_v_bits": ("uint32", int(qkv["v_bits"])),
        "offload.qkv_group_size": ("uint32", int(qkv["group_size"])),
        "offload.qkv_page_size_tokens": ("uint32", int(qkv["page_size_tokens"])),
        "offload.qkv_sink_tokens": ("uint32", int(qkv["residency"]["sink_tokens"])),
        "offload.anchor_layers": ("string", json_compact(residency["anchors"]["layers"])),
        "offload.valid_split_points": ("string", json_compact(residency["valid_split_points"])),
        "offload.reentry_points": ("string", json_compact(residency["reentry_points"])),
        "offload.pipeline_barrier_points": ("string", json_compact(execution["pipeline_barriers"]["points"])),
    }
    if "file_index" in contract:
        scalar_entries["offload.file_index"] = ("uint32", int(contract["file_index"]))
    if "file_count" in contract:
        scalar_entries["offload.file_count"] = ("uint32", int(contract["file_count"]))
    if "target_file" in contract:
        scalar_entries["offload.target_file"] = ("string", str(contract["target_file"]))
    complex_blocks = {
        "offload.metadata_v2": contract,
        "offload.arch_meta": contract["arch_meta"],
        "offload.weight_quant_schema": contract["weight_quant_schema"],
        "offload.qkv_cache_schema": contract["qkv_cache_schema"],
        "offload.qkv_aux_data": contract["qkv_aux_data"],
        "offload.residency_policy": contract["residency_policy"],
        "offload.execution_hints": contract["execution_hints"],
        "offload.dynamic_swap_triggers": contract["dynamic_swap_triggers"],
        "offload.memory_management_hints": contract["memory_management_hints"],
        "offload.prefetch_plan_hints": contract["prefetch_plan_hints"],
        "offload.kernel_hints": contract["kernel_hints"],
        "offload.kernel_config_hints": contract["kernel_config_hints"],
        "offload.cpu_execution_hints": contract["cpu_execution_hints"],
        "offload.context_length_table": contract["context_length_table"],
        "offload.io_schedule": contract["io_schedule"],
        "offload.streaming_attention": contract["streaming_attention"],
        "offload.numerical_stability": contract["numerical_stability"],
        "offload.continuous_batching": contract["continuous_batching"],
        "offload.batch_schedule": contract["batch_schedule"],
        "offload.tensor_layout_schema": contract["tensor_layout_schema"],
        "offload.integrity_schema": contract["integrity_schema"],
        "offload.per_layer_swap_cost": contract["per_layer_swap_cost"],
        "offload.runtime_monitoring_hints": contract["runtime_monitoring_hints"],
        "offload.thermal_power_hints": contract["thermal_power_hints"],
        "offload.speculative_execution_schema": contract["speculative_execution_schema"],
        "offload.activation_checkpoint_schema": contract["activation_checkpoint_schema"],
        "offload.mig_hints": contract["mig_hints"],
        "offload.profiling_measured_data": contract["profiling_measured_data"],
        "offload.multi_accelerator_detail": contract["multi_accelerator_detail"],
        "offload.compatibility": contract["compatibility"],
        "offload.runtime_note": contract["runtime_note"],
        "offload.sglang_runtime_note": contract["sglang_runtime_note"],
        "offload.engine_contract": contract["engine_contract"],
        "offload.backend_translation_contract": contract["backend_translation_contract"],
    }
    entries = dict(scalar_entries)
    for key, value in complex_blocks.items():
        entries[key] = ("string", json_compact(value))
    for key in ("container_contract", "part_contract_coverage", "current_file_metadata", "shard_set_manifest", "files_present"):
        if key in contract:
            entries[f"offload.{key}"] = ("string", json_compact(contract[key]))
    if contract.get("source_gguf_meta"):
        entries["offload.source_gguf_meta"] = ("string", json_compact(contract["source_gguf_meta"]))
    return entries


def gguf_strategy_json(entries, include_metadata_v2):
    out = {}
    for key, (dtype, value) in entries.items():
        if not include_metadata_v2 and key == "offload.metadata_v2":
            continue
        out[key] = {"type": dtype, "value": value}
    return out


def build_safetensors_metadata(contract):
    entries = build_gguf_kv_entries(contract)
    meta = {
        "format": "pt",
        "offload.format_version": "2",
        "offload.backend_neutral": "true",
    }
    for key, (_dtype, value) in entries.items():
        if key in {"offload.format_version", "offload.backend_neutral"}:
            continue
        meta[key] = value if isinstance(value, str) else json_compact(value)
    return {"__metadata__": meta}


def write_foreign_metadata_exports_for_stem(contract, stem):
    entries = build_gguf_kv_entries(contract)
    paths = {}
    paths["gguf_strategy_a"] = GGUF_EXPORT_DIR / f"{stem}.gguf-kv.strategy_a.json"
    paths["gguf_strategy_b"] = GGUF_EXPORT_DIR / f"{stem}.gguf-kv.strategy_b.json"
    paths["sidecar"] = SIDECAR_EXPORT_DIR / f"{stem}.offload.json"
    paths["safetensors_metadata"] = SAFETENSORS_EXPORT_DIR / f"{stem}.safetensors-metadata.json"

    paths["gguf_strategy_a"].write_text(json.dumps({
        "strategy": "A",
        "description": "GGUF native scalar KVs plus JSON string blocks",
        "kv": gguf_strategy_json(entries, include_metadata_v2=False),
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    paths["gguf_strategy_b"].write_text(json.dumps({
        "strategy": "B",
        "description": "single offload.metadata_v2 JSON plus format scalar",
        "kv": {
            "offload.metadata_v2": {"type": "string", "value": entries["offload.metadata_v2"][1]},
            "offload.format_version": {"type": "uint32", "value": 2},
            "offload.backend_neutral": {"type": "bool", "value": True},
        },
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    paths["sidecar"].write_text(json.dumps(contract, ensure_ascii=False, indent=2), encoding="utf-8")
    paths["safetensors_metadata"].write_text(json.dumps(build_safetensors_metadata(contract), ensure_ascii=False, indent=2), encoding="utf-8")
    return paths


GGUF_TYPE_UINT8 = 0
GGUF_TYPE_INT8 = 1
GGUF_TYPE_UINT16 = 2
GGUF_TYPE_INT16 = 3
GGUF_TYPE_UINT32 = 4
GGUF_TYPE_INT32 = 5
GGUF_TYPE_FLOAT32 = 6
GGUF_TYPE_BOOL = 7
GGUF_TYPE_STRING = 8
GGUF_TYPE_ARRAY = 9
GGUF_TYPE_UINT64 = 10
GGUF_TYPE_INT64 = 11
GGUF_TYPE_FLOAT64 = 12
GGUF_TYPE_SIZE = {
    GGUF_TYPE_UINT8: 1,
    GGUF_TYPE_INT8: 1,
    GGUF_TYPE_UINT16: 2,
    GGUF_TYPE_INT16: 2,
    GGUF_TYPE_UINT32: 4,
    GGUF_TYPE_INT32: 4,
    GGUF_TYPE_FLOAT32: 4,
    GGUF_TYPE_BOOL: 1,
    GGUF_TYPE_UINT64: 8,
    GGUF_TYPE_INT64: 8,
    GGUF_TYPE_FLOAT64: 8,
}


def gguf_read_exact(handle, size):
    data = handle.read(size)
    if len(data) != size:
        raise EOFError("unexpected EOF while reading GGUF")
    return data


def gguf_read_u32(handle):
    return struct.unpack("<I", gguf_read_exact(handle, 4))[0]


def gguf_read_u64(handle):
    return struct.unpack("<Q", gguf_read_exact(handle, 8))[0]


def gguf_read_string_value(handle):
    size = gguf_read_u64(handle)
    return gguf_read_exact(handle, size).decode("utf-8")


def gguf_skip_value(handle, value_type):
    if value_type == GGUF_TYPE_STRING:
        size = gguf_read_u64(handle)
        handle.seek(size, 1)
        return
    if value_type == GGUF_TYPE_ARRAY:
        elem_type = gguf_read_u32(handle)
        count = gguf_read_u64(handle)
        if elem_type == GGUF_TYPE_STRING:
            for _ in range(count):
                size = gguf_read_u64(handle)
                handle.seek(size, 1)
            return
        elem_size = GGUF_TYPE_SIZE.get(elem_type)
        if elem_size is None:
            raise ValueError(f"unsupported GGUF array element type: {elem_type}")
        handle.seek(elem_size * count, 1)
        return
    size = GGUF_TYPE_SIZE.get(value_type)
    if size is None:
        raise ValueError(f"unsupported GGUF value type: {value_type}")
    handle.seek(size, 1)


def gguf_skip_tensor_infos(handle, tensor_count):
    for _ in range(tensor_count):
        _name = gguf_read_string_value(handle)
        dims = gguf_read_u32(handle)
        handle.seek(8 * dims, 1)
        handle.seek(4, 1)
        handle.seek(8, 1)


def gguf_encode_string(text):
    raw = str(text).encode("utf-8")
    return struct.pack("<Q", len(raw)) + raw


def gguf_encode_kv(key, dtype, value):
    key_raw = gguf_encode_string(key)
    if dtype == "uint32":
        return key_raw + struct.pack("<I", GGUF_TYPE_UINT32) + struct.pack("<I", int(value))
    if dtype == "bool":
        return key_raw + struct.pack("<I", GGUF_TYPE_BOOL) + struct.pack("<?", bool(value))
    if dtype == "string":
        return key_raw + struct.pack("<I", GGUF_TYPE_STRING) + gguf_encode_string(value)
    raise ValueError(f"unsupported GGUF export dtype: {dtype}")


def patch_gguf_file_with_offload_kv(src_path, dst_path, contract, strategy="A"):
    src_path = Path(src_path)
    dst_path = Path(dst_path)
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    entries = build_gguf_kv_entries(contract)
    if strategy.upper() == "B":
        entries = {
            "offload.metadata_v2": entries["offload.metadata_v2"],
            "offload.format_version": ("uint32", 2),
            "offload.backend_neutral": ("bool", True),
        }

    retained_kv_raw = []
    alignment = 32
    with src_path.open("rb") as src:
        magic = gguf_read_exact(src, 4)
        if magic != b"GGUF":
            raise ValueError(f"not a GGUF file: {src_path}")
        version = gguf_read_u32(src)
        tensor_count = gguf_read_u64(src)
        kv_count = gguf_read_u64(src)
        for _ in range(kv_count):
            start = src.tell()
            key = gguf_read_string_value(src)
            value_type = gguf_read_u32(src)
            if key == "general.alignment" and value_type == GGUF_TYPE_UINT32:
                alignment = struct.unpack("<I", gguf_read_exact(src, 4))[0]
            else:
                gguf_skip_value(src, value_type)
            end = src.tell()
            if not key.startswith("offload."):
                src.seek(start)
                retained_kv_raw.append(src.read(end - start))
            src.seek(end)
        tensor_info_start = src.tell()
        gguf_skip_tensor_infos(src, tensor_count)
        tensor_info_end = src.tell()
        data_start = ((tensor_info_end + alignment - 1) // alignment) * alignment
        src.seek(tensor_info_start)
        tensor_info_raw = src.read(tensor_info_end - tensor_info_start)

        with dst_path.open("wb") as dst:
            dst.write(b"GGUF")
            dst.write(struct.pack("<I", version))
            dst.write(struct.pack("<Q", tensor_count))
            dst.write(struct.pack("<Q", len(retained_kv_raw) + len(entries)))
            for raw in retained_kv_raw:
                dst.write(raw)
            for key, (dtype, value) in entries.items():
                dst.write(gguf_encode_kv(key, dtype, value))
            dst.write(tensor_info_raw)
            pad = ((dst.tell() + alignment - 1) // alignment) * alignment - dst.tell()
            if pad:
                dst.write(b"\x00" * pad)
            src.seek(data_start)
            shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
    return {
        "path": str(dst_path),
        "bytes": dst_path.stat().st_size,
        "file_sha256": sha256_file(dst_path),
        "strategy": strategy.upper(),
        "kv_added": len(entries),
    }


def parse_content_range_total(header_value):
    if not header_value:
        return None
    try:
        return int(str(header_value).rsplit("/", 1)[1])
    except Exception:
        return None


def fetch_http_range(session, url, start, end=None, token=None, stream=False):
    headers = {"Range": f"bytes={start}-{'' if end is None else end}"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    resp = session.get(url, headers=headers, stream=stream, timeout=120)
    if resp.status_code != 206:
        resp.close()
        raise RuntimeError(f"server did not honor HTTP range request: status={resp.status_code} url={url}")
    return resp


def remote_file_size(session, url, token=None):
    resp = fetch_http_range(session, url, 0, 0, token=token, stream=False)
    try:
        total = parse_content_range_total(resp.headers.get("Content-Range"))
        if total is None:
            raise RuntimeError(f"missing total size in Content-Range for {url}")
        return total
    finally:
        resp.close()


def resolve_hf_download(spec, token=None):
    from huggingface_hub import get_hf_file_metadata
    hub_url = source_gguf_hub_url(spec)
    meta = get_hf_file_metadata(hub_url, token=token, timeout=60)
    return {
        "hub_url": hub_url,
        "download_url": getattr(meta, "location", None) or hub_url,
        "size": int(getattr(meta, "size", 0) or 0),
        "etag": getattr(meta, "etag", None),
        "commit_hash": getattr(meta, "commit_hash", None),
    }


def parse_gguf_prefix(prefix_raw, contract, strategy="A"):
    src = io.BytesIO(prefix_raw)
    entries = build_gguf_kv_entries(contract)
    if strategy.upper() == "B":
        entries = {
            "offload.metadata_v2": entries["offload.metadata_v2"],
            "offload.format_version": ("uint32", 2),
            "offload.backend_neutral": ("bool", True),
        }

    retained_kv_raw = []
    source_gguf_meta = {}
    alignment = 32
    magic = gguf_read_exact(src, 4)
    if magic != b"GGUF":
        raise ValueError("not a GGUF prefix")
    version = gguf_read_u32(src)
    tensor_count = gguf_read_u64(src)
    kv_count = gguf_read_u64(src)
    for _ in range(kv_count):
        start = src.tell()
        key = gguf_read_string_value(src)
        value_type = gguf_read_u32(src)
        if key == "general.alignment" and value_type == GGUF_TYPE_UINT32:
            value = struct.unpack("<I", gguf_read_exact(src, 4))[0]
            alignment = value
        elif any(key.startswith(prefix) for prefix in GGUF_ARCH_META_PREFIXES):
            value = gguf_decode_value(src, value_type)
            source_gguf_meta[key] = value
        else:
            gguf_skip_value(src, value_type)
        end = src.tell()
        if not key.startswith("offload."):
            src.seek(start)
            retained_kv_raw.append(src.read(end - start))
        src.seek(end)
    tensor_info_start = src.tell()
    gguf_skip_tensor_infos(src, tensor_count)
    tensor_info_end = src.tell()
    data_start = ((tensor_info_end + alignment - 1) // alignment) * alignment
    if data_start > len(prefix_raw):
        raise EOFError(f"GGUF header/tensor table needs {data_start} bytes, got {len(prefix_raw)}")
    src.seek(tensor_info_start)
    tensor_info_raw = src.read(tensor_info_end - tensor_info_start)

    out = io.BytesIO()
    out.write(b"GGUF")
    out.write(struct.pack("<I", version))
    out.write(struct.pack("<Q", tensor_count))
    out.write(struct.pack("<Q", len(retained_kv_raw) + len(entries)))
    for raw in retained_kv_raw:
        out.write(raw)
    for key, (dtype, value) in entries.items():
        out.write(gguf_encode_kv(key, dtype, value))
    out.write(tensor_info_raw)
    pad = ((out.tell() + alignment - 1) // alignment) * alignment - out.tell()
    if pad:
        out.write(b"\x00" * pad)
    return {
        "patched_prefix": out.getvalue(),
        "old_data_start": data_start,
        "kv_added": len(entries),
        "source_gguf_meta": source_gguf_meta,
    }


def patch_gguf_remote_file_to_local(url, dst_path, contract, token=None, strategy="A"):
    import requests
    dst_path = Path(dst_path)
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    header_bytes = GGUF_HEADER_RANGE_BYTES
    parsed = None
    with requests.Session() as session:
        total_bytes = remote_file_size(session, url, token=token)
        for _ in range(6):
            end = min(header_bytes, total_bytes) - 1
            resp = fetch_http_range(session, url, 0, end, token=token, stream=False)
            prefix_raw = resp.content
            resp.close()
            try:
                parsed = parse_gguf_prefix(prefix_raw, contract, strategy=strategy)
                break
            except EOFError:
                if header_bytes >= total_bytes:
                    raise
                header_bytes *= 2
        if parsed is None:
            raise RuntimeError("could not parse GGUF header/tensor table from ranged prefix")

        old_data_start = int(parsed["old_data_start"])
        source_hash = hashlib.sha256()
        output_hash = hashlib.sha256()
        source_hash.update(prefix_raw[:old_data_start])
        output_hash.update(parsed["patched_prefix"])
        streamed = 0
        last_print = 0
        with dst_path.open("wb") as out:
            out.write(parsed["patched_prefix"])
            data_resp = fetch_http_range(session, url, old_data_start, None, token=token, stream=True)
            try:
                for chunk in data_resp.iter_content(chunk_size=HTTP_CHUNK_BYTES):
                    if not chunk:
                        continue
                    out.write(chunk)
                    source_hash.update(chunk)
                    output_hash.update(chunk)
                    streamed += len(chunk)
                    if streamed - last_print >= PROGRESS_INTERVAL_BYTES:
                        if total_bytes:
                            done = old_data_start + streamed
                            print(f"stream patch: {done / (1024**3):.2f}/{total_bytes / (1024**3):.2f} GiB")
                        else:
                            print(f"stream patch: {(old_data_start + streamed) / (1024**3):.2f} GiB")
                        last_print = streamed
            finally:
                data_resp.close()

    source_bytes = (total_bytes or old_data_start + streamed)
    output_bytes = len(parsed["patched_prefix"]) + streamed
    return {
        "path": str(dst_path),
        "bytes": output_bytes,
        "file_sha256": output_hash.hexdigest(),
        "source_bytes": source_bytes,
        "source_sha256": source_hash.hexdigest(),
        "source_data_start": old_data_start,
        "patched_prefix_bytes": len(parsed["patched_prefix"]),
        "strategy": strategy.upper(),
        "kv_added": parsed["kv_added"],
        "source_gguf_meta": parsed["source_gguf_meta"],
        "storage_mode": "remote_range_to_single_local_output",
    }


def patch_gguf_bytes_to_local(source_bytes_data, dst_path, contract, strategy="A"):
    dst_path = Path(dst_path)
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    parsed = parse_gguf_prefix(source_bytes_data, contract, strategy=strategy)
    old_data_start = int(parsed["old_data_start"])
    output_hash = hashlib.sha256()
    output_hash.update(parsed["patched_prefix"])
    output_hash.update(source_bytes_data[old_data_start:])
    with dst_path.open("wb") as out:
        out.write(parsed["patched_prefix"])
        out.write(source_bytes_data[old_data_start:])
    return {
        "path": str(dst_path),
        "bytes": dst_path.stat().st_size,
        "file_sha256": output_hash.hexdigest(),
        "source_bytes": len(source_bytes_data),
        "source_sha256": hashlib.sha256(source_bytes_data).hexdigest(),
        "source_data_start": old_data_start,
        "patched_prefix_bytes": len(parsed["patched_prefix"]),
        "strategy": strategy.upper(),
        "kv_added": parsed["kv_added"],
        "source_gguf_meta": parsed["source_gguf_meta"],
        "storage_mode": "small_file_direct_get_to_single_local_output",
    }


def patch_gguf_hf_file_to_local(spec, dst_path, contract, token=None, strategy="A"):
    import requests
    info = resolve_hf_download(spec, token=token)
    signed_url = info["download_url"]
    size = int(info.get("size") or 0)
    try:
        return patch_gguf_remote_file_to_local(signed_url, dst_path, contract, token=None, strategy=strategy)
    except RuntimeError as exc:
        if "status=416" not in str(exc) or size <= 0 or size > SMALL_FILE_DIRECT_GET_LIMIT_BYTES:
            raise
        print(f"Range unavailable for small GGUF ({size} bytes); using direct GET fallback")
        resp = requests.get(signed_url, timeout=120)
        resp.raise_for_status()
        return patch_gguf_bytes_to_local(resp.content, dst_path, contract, strategy=strategy)


def read_remote_gguf_source_meta(url, contract, token=None, total_bytes_hint=0):
    import requests
    header_bytes = GGUF_HEADER_RANGE_BYTES
    with requests.Session() as session:
        total_bytes = int(total_bytes_hint or 0) or remote_file_size(session, url, token=token)
        for _ in range(6):
            end = min(header_bytes, total_bytes) - 1
            resp = fetch_http_range(session, url, 0, end, token=token, stream=False)
            prefix_raw = resp.content
            resp.close()
            try:
                return parse_gguf_prefix(prefix_raw, contract, strategy="A")["source_gguf_meta"]
            except EOFError:
                if header_bytes >= total_bytes:
                    raise
                header_bytes *= 2
    raise RuntimeError("could not read GGUF source metadata from ranged prefix")


def read_hf_gguf_source_meta(spec, contract, token=None):
    info = resolve_hf_download(spec, token=token)
    try:
        return read_remote_gguf_source_meta(info["download_url"], contract, token=None, total_bytes_hint=info.get("size") or 0)
    except RuntimeError as exc:
        size = int(info.get("size") or 0)
        if "status=416" not in str(exc) or size <= 0 or size > SMALL_FILE_DIRECT_GET_LIMIT_BYTES:
            raise
        import requests
        print(f"Range unavailable for small GGUF metadata read ({size} bytes); using direct GET fallback")
        resp = requests.get(info["download_url"], timeout=120)
        resp.raise_for_status()
        return parse_gguf_prefix(resp.content, contract, strategy="A")["source_gguf_meta"]


def gguf_decode_scalar(handle, value_type):
    if value_type == GGUF_TYPE_STRING:
        return gguf_read_string_value(handle)
    scalar_formats = {
        GGUF_TYPE_UINT8: "<B",
        GGUF_TYPE_INT8: "<b",
        GGUF_TYPE_UINT16: "<H",
        GGUF_TYPE_INT16: "<h",
        GGUF_TYPE_UINT32: "<I",
        GGUF_TYPE_INT32: "<i",
        GGUF_TYPE_FLOAT32: "<f",
        GGUF_TYPE_BOOL: "<?",
        GGUF_TYPE_UINT64: "<Q",
        GGUF_TYPE_INT64: "<q",
        GGUF_TYPE_FLOAT64: "<d",
    }
    fmt = scalar_formats.get(value_type)
    if fmt is None:
        raise ValueError(f"unsupported GGUF scalar type: {value_type}")
    return struct.unpack(fmt, gguf_read_exact(handle, GGUF_TYPE_SIZE[value_type]))[0]


def gguf_decode_value(handle, value_type):
    if value_type == GGUF_TYPE_ARRAY:
        elem_type = gguf_read_u32(handle)
        count = gguf_read_u64(handle)
        return [gguf_decode_scalar(handle, elem_type) for _ in range(count)]
    return gguf_decode_scalar(handle, value_type)


def read_gguf_kv(model_path, prefix="offload."):
    model_path = Path(model_path)
    out = {}

    def wanted(key):
        if prefix is None:
            return True
        if isinstance(prefix, (tuple, list, set)):
            return any(key.startswith(item) for item in prefix)
        return key.startswith(prefix)

    with model_path.open("rb") as handle:
        magic = gguf_read_exact(handle, 4)
        if magic != b"GGUF":
            raise ValueError(f"not a GGUF file: {model_path}")
        _version = gguf_read_u32(handle)
        _tensor_count = gguf_read_u64(handle)
        kv_count = gguf_read_u64(handle)
        for _ in range(kv_count):
            key = gguf_read_string_value(handle)
            value_type = gguf_read_u32(handle)
            if wanted(key):
                out[key] = gguf_decode_value(handle, value_type)
            else:
                gguf_skip_value(handle, value_type)
    return out


def parse_json_if_possible(value):
    if not isinstance(value, str):
        return value
    try:
        return json.loads(value)
    except json.JSONDecodeError:
        return value


def normalize_offload_metadata(meta, prefer_metadata_v2=True):
    offload = {}
    for key, value in dict(meta).items():
        if not key.startswith("offload."):
            continue
        field = key[len("offload."):]
        offload[field] = parse_json_if_possible(value)
    if prefer_metadata_v2 and isinstance(offload.get("metadata_v2"), dict):
        merged = dict(offload["metadata_v2"])
        for key, value in offload.items():
            if key != "metadata_v2" and key not in merged:
                merged[key] = value
        return merged
    return offload


def read_safetensors_metadata(model_path):
    model_path = Path(model_path)
    with model_path.open("rb") as handle:
        header_size = struct.unpack("<Q", gguf_read_exact(handle, 8))[0]
        header = json.loads(gguf_read_exact(handle, header_size).decode("utf-8"))
    return header.get("__metadata__", {})


def find_offload_sidecar(model_path):
    model_path = Path(model_path)
    stem = model_path.stem
    parent = model_path.parent
    candidates = [
        parent / f"{stem}.offload.json",
        parent / "offload_meta.json",
        parent / "metadata" / "sidecar" / f"{stem}.offload.json",
        parent.parent / "metadata" / "sidecar" / f"{stem}.offload.json",
    ]
    for candidate in candidates:
        if not candidate.exists():
            continue
        try:
            return json.loads(candidate.read_text(encoding="utf-8"))
        except (UnicodeDecodeError, json.JSONDecodeError):
            continue
    return None


def load_offload_meta(model_path, prefer_sidecar=False, prefer_metadata_v2=True):
    model_path = Path(model_path)
    sidecar = find_offload_sidecar(model_path)
    if prefer_sidecar and sidecar is not None:
        return sidecar

    ext = model_path.suffix.lower()
    meta = {}
    if ext == ".gguf":
        meta = read_gguf_kv(model_path, prefix="offload.")
    elif ext == ".safetensors":
        meta = read_safetensors_metadata(model_path)
    elif ext == ".json":
        return json.loads(model_path.read_text(encoding="utf-8"))

    offload = normalize_offload_metadata(meta, prefer_metadata_v2=prefer_metadata_v2)
    if offload:
        return offload
    if sidecar is not None:
        return sidecar
    return {}


def build_arch_meta(source_gguf_meta=None):
    arch = dict(MODEL_PROFILE["arch_meta"])
    source_gguf_meta = dict(source_gguf_meta or {})
    if source_gguf_meta.get("general.architecture"):
        arch["model_family"] = source_gguf_meta["general.architecture"]
    key_map = {
        "block_count": "n_layers",
        "attention.head_count": "n_heads",
        "attention.head_count_kv": "n_kv_heads",
        "attention.key_length": "head_dim",
        "attention.value_length": "head_dim",
        "embedding_length": "hidden_dim",
        "feed_forward_length": "ffn_dim",
        "context_length": "max_seq_len",
    }
    for key, value in source_gguf_meta.items():
        suffix = key.split(".", 1)[-1]
        if suffix in key_map:
            arch[key_map[suffix]] = value
    arch["source_gguf_metadata_keys"] = sorted(source_gguf_meta.keys())
    return arch


def build_gguf_file_contract(file_spec, source_gguf_meta=None):
    idx = int(file_spec["index"])
    count = int(file_spec["count"])
    source_gguf_meta = dict(source_gguf_meta or {})
    current_file = {
        "index": idx,
        "count": count,
        "name": file_spec["name"],
        "source_path": file_spec["source_path"],
        "target_path": target_gguf_repo_path(file_spec),
    }
    all_files = [
        {
            "index": int(spec["index"]),
            "count": int(spec["count"]),
            "name": spec["name"],
            "source_path": spec["source_path"],
            "target_path": target_gguf_repo_path(spec),
        }
        for spec in GGUF_FILE_SPECS
    ]
    return {
        "format_version": 2,
        "contract_name": "StorageLLM JUJU v2 GGUF offload-native metadata",
        "patched_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
        "notebook_build_id": NOTEBOOK_BUILD_ID,
        "artifact_format": "gguf",
        "source_repo": SOURCE_HF_REPO_ID,
        "source_subdir": SOURCE_SUBDIR,
        "source_file": file_spec["source_path"],
        "source_gguf_meta": source_gguf_meta,
        "source_weight_quant_family": MODEL_PROFILE["weight_quant_schema"]["source_family"],
        "source_weight_kernel_family": MODEL_PROFILE["weight_quant_schema"]["source_kernel_family"],
        "source_weight_bits": MODEL_PROFILE["weight_quant_schema"]["source_bits"],
        "source_weight_block_size": MODEL_PROFILE["weight_quant_schema"]["source_block_size"],
        "target_repo": TARGET_HF_REPO_ID,
        "target_file": target_gguf_repo_path(file_spec),
        "file_index": idx,
        "file_count": count,
        "container_contract": {
            "primary_metadata_location": "gguf_kv_offload_namespace",
            "sidecar_required_for_runtime": False,
            "folder_name_runtime_dependency": False,
            "file_name_runtime_dependency": "only_for_shard_order_discovery",
            "runtime_should_read": ["offload.metadata_v2", "offload.file_index", "offload.file_count"],
        },
        "files_present": list(range(1, count + 1)),
        "current_file_metadata": current_file,
        "shard_set_manifest": {
            "runtime_lookup_key": "file_index",
            "file_count": count,
            "files": all_files,
        },
        "part_contract_coverage": {
            "scope": "full_11_file_gguf_shard_set",
            "current_file_index": idx,
            "current_file_name": file_spec["name"],
            "required_file_count": count,
            "all_file_names_embedded": True,
            "per_file_contract_repeated": True,
            "metadata_repetition_reason": "each GGUF shard remains self-describing when downloaded alone",
            "runtime_selection_rule": "use offload.file_index/current_file_metadata for this shard and shard_set_manifest.files for sibling lookup",
        },
        "hardware_topology": {
            "embedded": False,
            "runtime_probe_required": True,
            "reason": "GGUF remains portable; engine probes VRAM, interconnect, CPU ISA, RAM, NVMe, and direct-storage support at startup",
        },
        "arch_meta": build_arch_meta(source_gguf_meta),
        "weight_quant_schema": MODEL_PROFILE["weight_quant_schema"],
        "qkv_cache_schema": MODEL_PROFILE["qkv_cache_schema"],
        "qkv_aux_data": get_qkv_aux(),
        "residency_policy": MODEL_PROFILE["residency_policy"],
        "prefetch_plan_hints": MODEL_PROFILE["prefetch_plan_hints"],
        "kernel_hints": MODEL_PROFILE["kernel_hints"],
        "execution_hints": MODEL_PROFILE["execution_hints"],
        "dynamic_swap_triggers": MODEL_PROFILE["dynamic_swap_triggers"],
        "memory_management_hints": MODEL_PROFILE["memory_management_hints"],
        "cpu_execution_hints": MODEL_PROFILE["cpu_execution_hints"],
        "kernel_config_hints": MODEL_PROFILE["kernel_config_hints"],
        "context_length_table": MODEL_PROFILE["context_length_table"],
        "io_schedule": MODEL_PROFILE["io_schedule"],
        "streaming_attention": MODEL_PROFILE["streaming_attention"],
        "numerical_stability": MODEL_PROFILE["numerical_stability"],
        "continuous_batching": MODEL_PROFILE["continuous_batching"],
        "batch_schedule": MODEL_PROFILE["batch_schedule"],
        "tensor_layout_schema": MODEL_PROFILE["tensor_layout_schema"],
        "tensor_layout_per_tensor": {
            "mode": "runtime_parse_gguf_tensor_infos",
            "source": "gguf_tensor_table",
            "file_index": idx,
            "layout_overrides": [],
        },
        "integrity_schema": MODEL_PROFILE["integrity_schema"],
        "integrity_per_layer": {
            "mode": "runtime_parse_gguf_tensor_table_and_file_sha256",
            "file_level_sha256_recorded_in_verify_json": True,
            "layer_level_checksums_runtime_derivable": True,
        },
        "per_layer_swap_cost": {
            "mode": "runtime_estimate_from_gguf_tensor_sizes",
            "transfer_time_model": "bytes_div_runtime_measured_bandwidth",
            "recompute_time_model": "runtime_profile_or_conservative_unknown",
            "recommendation": "runtime_compare_transfer_vs_recompute",
        },
        "activation_stats_schema": MODEL_PROFILE["activation_stats_schema"],
        "sparsity_schema": MODEL_PROFILE["sparsity_schema"],
        "runtime_monitoring_hints": MODEL_PROFILE["runtime_monitoring_hints"],
        "thermal_power_hints": MODEL_PROFILE["thermal_power_hints"],
        "speculative_execution_schema": MODEL_PROFILE["speculative_execution_schema"],
        "activation_checkpoint_schema": MODEL_PROFILE["activation_checkpoint_schema"],
        "mig_hints": MODEL_PROFILE["mig_hints"],
        "profiling_measured_data": {
            "present": False,
            "schema": "runtime_updates_measured_per_layer_data_after_first_profile_run",
            "runtime_update_allowed": True,
        },
        "multi_accelerator_detail": MODEL_PROFILE["multi_accelerator_detail"],
        "foreign_container_export": MODEL_PROFILE["foreign_container_export"],
        "backend_translation_contract": MODEL_PROFILE["backend_translation_contract"],
        "compatibility": MODEL_PROFILE["compatibility"],
        "runtime_note": {
            "required_flags": REQUIRED_RUNTIME_FLAGS,
            "reason": "model_specific_flags_from_env_or_source_notes",
        },
        "sglang_runtime_note": {
            "required_flag": REQUIRED_RUNTIME_FLAGS[0] if REQUIRED_RUNTIME_FLAGS else "",
            "required_flags": REQUIRED_RUNTIME_FLAGS,
            "reason": "model_specific_sglang_flags_from_env_or_source_notes",
        },
        "engine_contract": {
            "persistent_plain_kv_cache_allowed": False,
            "projection_kv_scratch_allowed": True,
            "qkv_packed_cache_required": True,
            "attention_must_read_packed_qkv": True,
            "offload_units": ["gguf_tensor_span", "expert_triplet", "qkv_page"],
            "shared_experts_fusion_allowed": SHARED_EXPERTS_FUSION_ALLOWED,
            "required_runtime_flags": REQUIRED_RUNTIME_FLAGS,
        },
    }


def validate_v2_contract(contract):
    errors = []
    if contract["hardware_topology"]["embedded"] is not False:
        errors.append("hardware topology must not be embedded")
    if contract["engine_contract"]["persistent_plain_kv_cache_allowed"] is not False:
        errors.append("persistent plain KV is still allowed")
    if contract["engine_contract"]["qkv_packed_cache_required"] is not True:
        errors.append("QKV packed cache is not required")
    required_flags = contract["engine_contract"].get("required_runtime_flags", [])
    if contract["engine_contract"]["shared_experts_fusion_allowed"] is False and not required_flags:
        errors.append("shared expert fusion is disabled but required_runtime_flags is empty")
    if contract["sglang_runtime_note"].get("required_flags", []) != required_flags:
        errors.append("sglang runtime note must mirror engine required_runtime_flags")
    required_sections = [
        "io_schedule",
        "streaming_attention",
        "numerical_stability",
        "per_layer_swap_cost",
        "continuous_batching",
        "batch_schedule",
        "tensor_layout_per_tensor",
        "integrity_per_layer",
        "mig_hints",
        "profiling_measured_data",
        "multi_accelerator_detail",
    ]
    for section in required_sections:
        if section not in contract:
            errors.append(f"missing required v2 section: {section}")
    residency = contract.get("residency_policy", {})
    if not residency.get("valid_split_points") or not residency.get("reentry_points"):
        errors.append("residency policy must include valid_split_points and reentry_points")
    cpu_hints = contract.get("cpu_execution_hints", {})
    if "numa_hints" not in cpu_hints:
        errors.append("cpu_execution_hints must include detailed numa_hints")
    forbidden_terms = ["cuda_graph", "cudaStream", "hipStream", "MTLCommandQueue", "VkQueue", "warp_count"]
    encoded = json.dumps(contract, ensure_ascii=False)
    for term in forbidden_terms:
        if term in encoded:
            errors.append(f"backend-specific term leaked into portable footer: {term}")
    optional = set(contract.get("compatibility", {}).get("optional_features", []))
    if "cuda_graph" in optional or "multi_gpu" in optional or "gds" in optional:
        errors.append("optional_features must use portable names, not CUDA/GPU-specific names")
    return errors


def release_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


def cleanup_paths(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            if path.is_dir():
                shutil.rmtree(path)
            else:
                path.unlink()
            print("Deleted local:", path)
    release_memory()


In [ ]:
HfApi, create_repo = ensure_hf_api()
HF_TOKEN = get_hf_token()
if UPLOAD_TO_HF and not HF_TOKEN:
    raise RuntimeError("Missing Hugging Face write token")

api = HfApi(token=HF_TOKEN)
if UPLOAD_TO_HF:
    create_repo(repo_id=TARGET_HF_REPO_ID, repo_type="model", private=False, exist_ok=True, token=HF_TOKEN)

model_card_path = MODEL_ROOT / "README.md"
model_card_path.write_text(f'''# {SOURCE_FILE_PREFIX} StorageLLM Offload

Source GGUF repo: https://huggingface.co/{SOURCE_HF_REPO_ID}/tree/main/{SOURCE_SUBDIR}

This repo contains GGUF shards patched with StorageLLM JUJU v2 offload metadata.
The tensor payload is preserved by streaming copy; the GGUF metadata table gets
additional `offload.*` keys for QKV packed cache, active-set residency,
portable execution hints, swap policy, streaming attention, batching,
integrity/profiling schemas, and backend-neutral runtime translation.

No separate runtime metadata file is required for the patched GGUF path. The
sidecar/strategy JSON files under `metadata/` are reference exports for other
containers and verification.

Runtime flags embedded in the GGUF metadata:
`{", ".join(REQUIRED_RUNTIME_FLAGS) if REQUIRED_RUNTIME_FLAGS else "none"}`.

Download note: the notebook sends `SOURCE_HF_TOKEN` when set, otherwise it uses
the same `HF_TOKEN` used for upload. This keeps the flow working if the source
repo becomes gated and the token has accepted access.
''', encoding="utf-8")
if UPLOAD_TO_HF:
    api.upload_file(
        path_or_fileobj=str(model_card_path),
        path_in_repo="README.md",
        repo_id=TARGET_HF_REPO_ID,
        repo_type="model",
        commit_message="Add StorageLLM offload model card",
    )

run_summary = {
    "ok": True,
    "errors": [],
    "notebook_build_id": NOTEBOOK_BUILD_ID,
    "source_repo_id": SOURCE_HF_REPO_ID,
    "source_subdir": SOURCE_SUBDIR,
    "source_file_prefix": SOURCE_FILE_PREFIX,
    "source_file_count": SOURCE_FILE_COUNT,
    "target_repo_id": TARGET_HF_REPO_ID,
    "target_subdir": TARGET_SUBDIR,
    "files": [],
    "started_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "continue_on_part_error": CONTINUE_ON_PART_ERROR,
}

source_download_token = os.environ.get("SOURCE_HF_TOKEN") or HF_TOKEN

for spec in FILES_TO_RUN:
    idx = int(spec["index"])
    name = spec["name"]
    local_out = OUT_DIR / name
    local_verify = VERIFY_DIR / gguf_verify_json_name(spec)
    export_paths = {}
    uploaded = False
    print("\\n========== GGUF", f"{idx:05d}/{SOURCE_FILE_COUNT:05d}", name, "==========")
    try:
        base_contract = build_gguf_file_contract(spec)
        source_gguf_meta = read_hf_gguf_source_meta(spec, base_contract, token=source_download_token)
        contract = build_gguf_file_contract(spec, source_gguf_meta=source_gguf_meta)
        errors = validate_v2_contract(contract)
        if errors:
            raise RuntimeError("contract validation failed: " + "; ".join(errors))

        if local_out.exists():
            local_out.unlink()
        write_info = patch_gguf_hf_file_to_local(spec, local_out, contract, token=source_download_token, strategy="A")
        write_info.pop("source_gguf_meta", None)
        verify = {
            "ok": True,
            "errors": [],
            "source_repo_id": SOURCE_HF_REPO_ID,
            "target_repo_id": TARGET_HF_REPO_ID,
            "file_index": idx,
            "file_count": int(spec["count"]),
            "source_path": spec["source_path"],
            "target_path": target_gguf_repo_path(spec),
            "source_gguf_meta_keys": sorted(source_gguf_meta.keys()),
            "input": {"path": source_gguf_hub_url(spec), "bytes": write_info["source_bytes"], "sha256": write_info["source_sha256"]},
            "output": write_info,
            "patch_mode": "remote_range_to_single_local_output",
            "contract": {
                "qkv_packed_cache_required": True,
                "persistent_plain_kv_cache_allowed": False,
                "shared_experts_fusion_allowed": SHARED_EXPERTS_FUSION_ALLOWED,
                "required_runtime_flags": REQUIRED_RUNTIME_FLAGS,
                "hardware_topology_embedded": False,
                "full_shard_set_contract_embedded": True,
            },
            "patched_at": contract["patched_at"],
        }
        local_verify.write_text(json.dumps(verify, ensure_ascii=False, indent=2), encoding="utf-8")
        if EXPORT_FOREIGN_METADATA:
            export_paths = write_foreign_metadata_exports_for_stem(contract, portable_stem_for_gguf(spec))

        if UPLOAD_TO_HF:
            repo_path = target_gguf_repo_path(spec)
            print("Uploading patched GGUF:", local_out, "->", repo_path)
            api.upload_file(
                path_or_fileobj=str(local_out),
                path_in_repo=repo_path,
                repo_id=TARGET_HF_REPO_ID,
                repo_type="model",
                commit_message=f"Patch offload GGUF shard {idx:05d}/{SOURCE_FILE_COUNT:05d}",
            )
            print("Uploading verify JSON:", local_verify, "->", f"verify/{local_verify.name}")
            api.upload_file(
                path_or_fileobj=str(local_verify),
                path_in_repo=f"verify/{local_verify.name}",
                repo_id=TARGET_HF_REPO_ID,
                repo_type="model",
                commit_message=f"Add GGUF offload verify {idx:05d}/{SOURCE_FILE_COUNT:05d}",
            )
            if UPLOAD_FOREIGN_METADATA and export_paths:
                for export_kind, export_path in export_paths.items():
                    if export_kind.startswith("gguf_"):
                        metadata_path = f"metadata/gguf/{export_path.name}"
                    elif export_kind == "safetensors_metadata":
                        metadata_path = f"metadata/safetensors/{export_path.name}"
                    else:
                        metadata_path = f"metadata/sidecar/{export_path.name}"
                    print("Uploading foreign metadata:", export_path, "->", metadata_path)
                    api.upload_file(
                        path_or_fileobj=str(export_path),
                        path_in_repo=metadata_path,
                        repo_id=TARGET_HF_REPO_ID,
                        repo_type="model",
                        commit_message=f"Add offload metadata export {export_kind} {idx:05d}/{SOURCE_FILE_COUNT:05d}",
                    )
            uploaded = True

        run_summary["files"].append({
            "index": idx,
            "name": name,
            "uploaded": uploaded,
            "target_path": target_gguf_repo_path(spec),
            "input_bytes": write_info["source_bytes"],
            "output_bytes": write_info["bytes"],
            "storage_mode": write_info["storage_mode"],
            "kv_added": write_info["kv_added"],
            "foreign_metadata_exports": {k: str(v) for k, v in export_paths.items()},
        })

        if DELETE_LOCAL_AFTER_UPLOAD and (uploaded or not UPLOAD_TO_HF):
            cleanup_paths([local_out, local_verify, *export_paths.values()])
    except Exception as exc:
        run_summary["ok"] = False
        err = {"index": idx, "name": name, "type": type(exc).__name__, "message": str(exc)[:1000]}
        run_summary["errors"].append(err)
        print("ERROR:", json.dumps(err, ensure_ascii=False))
        if not KEEP_FAILED_LOCAL_FILES:
            cleanup_paths([p for p in [local_out, *export_paths.values()] if p])
        if not CONTINUE_ON_PART_ERROR:
            raise
    finally:
        release_memory()

run_summary["finished_at"] = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())
summary_stem = re.sub(r"[^A-Za-z0-9._-]+", "_", SOURCE_FILE_PREFIX).strip("_") or "gguf"
summary_path = VERIFY_DIR / f"{summary_stem}_offload_patch_run_summary.json"
summary_path.write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(run_summary, ensure_ascii=False, indent=2))
print("RUN_SUMMARY:", summary_path)
